In [10]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, classification_report
from tqdm import tqdm
import re

class ProjectDataset(Dataset):
    def __init__(self, descriptions, targets, tokenizer, max_length=128):
        self.descriptions = descriptions
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.descriptions)
    
    def __getitem__(self, idx):
        description = str(self.descriptions[idx])
        target = self.targets[idx]
        
        encoding = self.tokenizer(
            description,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(target)
        }

class TransformerSkillPredictor(nn.Module):
    def __init__(self, n_classes, model_name='bert-base-uncased'):
        super(TransformerSkillPredictor, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)

class DeepSkillPredictor:
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.mlb = MultiLabelBinarizer()
        self.model = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_losses = []
        self.val_losses = []
        
    def prepare_data(self, df, test_size=0.2, random_state=42):
        """准备数据并划分训练集和测试集"""
        descriptions = df['description'].tolist()
        
        skills_list = df['required_skillsets'].apply(
            lambda x: [skill.strip() for skill in str(x).split(',') if skill.strip()]
        ).tolist()
        
        # 转换标签
        y = self.mlb.fit_transform(skills_list)
        
        # 划分训练集和测试集
        X_train, X_test, y_train, y_test = train_test_split(
            descriptions, y, test_size=test_size, random_state=random_state, stratify=None
        )
        
        return X_train, X_test, y_train, y_test
    
    def train(self, df, epochs=3, batch_size=16, learning_rate=2e-5, test_size=0.2):
        """训练模型，包含训练集和验证集划分"""
        # 准备数据并划分
        X_train, X_val, y_train, y_val = self.prepare_data(df, test_size=test_size)
        
        # 创建数据集
        train_dataset = ProjectDataset(X_train, y_train, self.tokenizer)
        val_dataset = ProjectDataset(X_val, y_val, self.tokenizer)
        
        # 创建数据加载器
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # 初始化模型
        self.model = TransformerSkillPredictor(len(self.mlb.classes_))
        self.model.to(self.device)
        
        # 优化器和损失函数
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        criterion = nn.BCEWithLogitsLoss()
        
        # 训练循环
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            # 训练阶段
            self.model.train()
            train_loss = 0
            train_progress = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
            
            for batch in train_progress:
                optimizer.zero_grad()
                
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                train_progress.set_postfix({'train_loss': f'{loss.item():.4f}'})
            
            avg_train_loss = train_loss / len(train_dataloader)
            self.train_losses.append(avg_train_loss)
            
            # 验证阶段
            self.model.eval()
            val_loss = 0
            all_preds = []
            all_labels = []
            
            val_progress = tqdm(val_dataloader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            
            with torch.no_grad():
                for batch in val_progress:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)
                    
                    outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item()
                    val_progress.set_postfix({'val_loss': f'{loss.item():.4f}'})
                    
                    # 收集预测结果用于评估
                    preds = torch.sigmoid(outputs).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(labels.cpu().numpy())
            
            avg_val_loss = val_loss / len(val_dataloader)
            self.val_losses.append(avg_val_loss)
            
            # 计算评估指标
            val_metrics = self._calculate_metrics(all_preds, all_labels)
            
            print(f'Epoch {epoch+1}:')
            print(f'  Train Loss: {avg_train_loss:.4f}')
            print(f'  Val Loss: {avg_val_loss:.4f}')
            print(f'  Val Hamming Loss: {val_metrics["hamming_loss"]:.4f}')
            print(f'  Val F1-Score: {val_metrics["f1_score"]:.4f}')
            
            # 简单的早停机制
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                # 可以在这里保存最佳模型
                torch.save(self.model.state_dict(), 'best_model.pth')
        
        print("训练完成!")
        return self.train_losses, self.val_losses
    
    def _calculate_metrics(self, preds, labels, threshold=0.3):
        """计算评估指标"""
        # 将概率转换为二进制预测
        binary_preds = (np.array(preds) > threshold).astype(int)
        binary_labels = np.array(labels)
        
        # 计算Hamming Loss
        h_loss = hamming_loss(binary_labels, binary_preds)
        
        # 计算F1-score (micro average)
        f1 = f1_score(binary_labels, binary_preds, average='micro', zero_division=0)
        
        return {
            'hamming_loss': h_loss,
            'f1_score': f1
        }
    
    def evaluate(self, df, batch_size=16, threshold=0.3):
        """在完整数据集上评估模型"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        descriptions = df['description'].tolist()
        
        skills_list = df['required_skillsets'].apply(
            lambda x: [skill.strip() for skill in str(x).split(',') if skill.strip()]
        ).tolist()
        
        # 转换标签
        y_true = self.mlb.transform(skills_list)
        
        # 创建数据集和数据加载器
        dataset = ProjectDataset(descriptions, y_true, self.tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        
        self.model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.sigmoid(outputs).cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        # 计算评估指标
        metrics = self._calculate_metrics(all_preds, all_labels, threshold)
        
        print("=== 模型评估结果 ===")
        print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")
        print(f"F1-Score: {metrics['f1_score']:.4f}")
        
        return metrics
    
    def predict_skills(self, description, threshold=0.3):
        """预测技能"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        self.model.eval()
        
        encoding = self.tokenizer(
            description,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        
        with torch.no_grad():
            input_ids = encoding['input_ids'].to(self.device)
            attention_mask = encoding['attention_mask'].to(self.device)
            
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            probabilities = torch.sigmoid(outputs).cpu().numpy()[0]
        
        # 获取预测的技能
        predicted_skills = []
        for i, prob in enumerate(probabilities):
            if prob > threshold:
                skill = self.mlb.classes_[i]
                predicted_skills.append((skill, prob))
        
        predicted_skills.sort(key=lambda x: x[1], reverse=True)
        
        return [skill for skill, prob in predicted_skills]
    
    def save_model(self, filepath):
        """保存模型"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'mlb': self.mlb,
            'tokenizer': self.tokenizer,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses
        }, filepath)
        print(f"模型已保存到 {filepath}")
    
    def load_model(self, filepath):
        """加载模型"""
        checkpoint = torch.load(filepath, map_location=self.device)
        self.mlb = checkpoint['mlb']
        self.tokenizer = checkpoint['tokenizer']
        self.train_losses = checkpoint.get('train_losses', [])
        self.val_losses = checkpoint.get('val_losses', [])
        
        self.model = TransformerSkillPredictor(len(self.mlb.classes_))
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        
        print(f"模型已从 {filepath} 加载")

def train_and_evaluate_model2():
    """训练并评估模型"""
    df = pd.read_csv('projects_with_skillsets.csv')
    print(f"数据集大小: {len(df)}")
    
    deep_predictor = DeepSkillPredictor(model_name='distilbert-base-uncased')
    
    print("开始训练模型...")
    train_losses, val_losses = deep_predictor.train(
        df, 
        epochs=100, 
        batch_size=8, 
        learning_rate=3e-5,
        test_size=0.2
    )
    
    deep_predictor.save_model('deep_skill_model.pth')
    
    print("\n在完整数据集上评估模型...")
    metrics = deep_predictor.evaluate(df)
    
    print("\n=== 示例预测 ===")
    test_descriptions = [
        "Develop a cloud-based machine learning platform for financial data analysis",
        "Build a responsive web application with React and JavaScript",
        "Create a data analytics dashboard with real-time visualization"
    ]
    
    for desc in test_descriptions:
        skills = deep_predictor.predict_skills(desc, threshold=0.2)
        print(f"描述: {desc}")
        print(f"预测技能: {skills}")
        print("-" * 50)
    
    return deep_predictor, metrics

# 运行训练和评估
if __name__ == "__main__":
    deep_predictor, metrics = train_and_evaluate_model2()

数据集大小: 1000
开始训练模型...


Epoch 1/100 [Val]: 100%|███████████████████████████████████████████████| 25/25 [00:00<00:00, 61.15it/s, val_loss=0.4238]


Epoch 1:
  Train Loss: 0.4801
  Val Loss: 0.4067
  Val Hamming Loss: 0.2421
  Val F1-Score: 0.7197


Epoch 2/100 [Val]: 100%|███████████████████████████████████████████████| 25/25 [00:00<00:00, 61.12it/s, val_loss=0.3033]


Epoch 2:
  Train Loss: 0.3591
  Val Loss: 0.3021
  Val Hamming Loss: 0.1358
  Val F1-Score: 0.8249


...

Epoch 99:
  Train Loss: 0.0096
  Val Loss: 0.0028
  Val Hamming Loss: 0.0013
  Val F1-Score: 0.9982


Epoch 100/100 [Val]: 100%|█████████████████████████████████████████████| 25/25 [00:00<00:00, 65.05it/s, val_loss=0.0007]


Epoch 100:
  Train Loss: 0.0040
  Val Loss: 0.0011
  Val Hamming Loss: 0.0000
  Val F1-Score: 1.0000
训练完成!
模型已保存到 deep_skill_model.pth

在完整数据集上评估模型...


Evaluating: 100%|███████████████████████████████████████████████████████████████████████| 63/63 [00:01<00:00, 42.67it/s]


=== 模型评估结果 ===
Hamming Loss: 0.0000
F1-Score: 1.0000

=== 示例预测 ===
描述: Develop a cloud-based machine learning platform for financial data analysis
预测技能: ['Python', 'Data Analysis', 'JavaScript', 'Machine Learning', 'AWS', 'React']
--------------------------------------------------
描述: Build a responsive web application with React and JavaScript
预测技能: ['React', 'JavaScript']
--------------------------------------------------
描述: Create a data analytics dashboard with real-time visualization
预测技能: ['Data Analysis', 'Python', 'React', 'JavaScript']
--------------------------------------------------


In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, classification_report
from tqdm import tqdm
import re
import os

class ProjectDataset(Dataset):
    def __init__(self, descriptions, targets, tokenizer, max_length=128):
        self.descriptions = descriptions
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.descriptions)
    
    def __getitem__(self, idx):
        description = str(self.descriptions[idx])
        target = self.targets[idx]
        
        encoding = self.tokenizer(
            description,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(target)
        }

class TransformerSkillPredictor(nn.Module):
    def __init__(self, n_classes, model_name='bert-base-uncased'):
        super(TransformerSkillPredictor, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)

class DeepSkillPredictor:
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.mlb = MultiLabelBinarizer()
        self.model = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_losses = []
        self.val_losses = []
        
    def prepare_data(self, df, test_size=0.2, random_state=42):
        """准备数据并划分训练集和测试集"""
        descriptions = df['description'].tolist()
        
        skills_list = df['required_skillsets'].apply(
            lambda x: [skill.strip() for skill in str(x).split(',') if skill.strip()]
        ).tolist()
        
        # 转换标签
        y = self.mlb.fit_transform(skills_list)
        
        # 划分训练集和测试集
        X_train, X_test, y_train, y_test = train_test_split(
            descriptions, y, test_size=test_size, random_state=random_state, stratify=None
        )
        
        return X_train, X_test, y_train, y_test
    
    def train(self, df, epochs=3, batch_size=16, learning_rate=2e-5, test_size=0.2):
        """训练模型，包含训练集和验证集划分"""
        # 准备数据并划分
        X_train, X_val, y_train, y_val = self.prepare_data(df, test_size=test_size)
        
        # 创建数据集
        train_dataset = ProjectDataset(X_train, y_train, self.tokenizer)
        val_dataset = ProjectDataset(X_val, y_val, self.tokenizer)
        
        # 创建数据加载器
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # 初始化模型
        self.model = TransformerSkillPredictor(len(self.mlb.classes_))
        self.model.to(self.device)
        
        # 优化器和损失函数
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        criterion = nn.BCEWithLogitsLoss()
        
        # 训练循环
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            # 训练阶段
            self.model.train()
            train_loss = 0
            train_progress = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
            
            for batch in train_progress:
                optimizer.zero_grad()
                
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                train_progress.set_postfix({'train_loss': f'{loss.item():.4f}'})
            
            avg_train_loss = train_loss / len(train_dataloader)
            self.train_losses.append(avg_train_loss)
            
            # 验证阶段
            self.model.eval()
            val_loss = 0
            all_preds = []
            all_labels = []
            
            val_progress = tqdm(val_dataloader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            
            with torch.no_grad():
                for batch in val_progress:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)
                    
                    outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item()
                    val_progress.set_postfix({'val_loss': f'{loss.item():.4f}'})
                    
                    # 收集预测结果用于评估
                    preds = torch.sigmoid(outputs).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(labels.cpu().numpy())
            
            avg_val_loss = val_loss / len(val_dataloader)
            self.val_losses.append(avg_val_loss)
            
            # 计算评估指标
            val_metrics = self._calculate_metrics(all_preds, all_labels)
            
            print(f'Epoch {epoch+1}:')
            print(f'  Train Loss: {avg_train_loss:.4f}')
            print(f'  Val Loss: {avg_val_loss:.4f}')
            print(f'  Val Hamming Loss: {val_metrics["hamming_loss"]:.4f}')
            print(f'  Val F1-Score: {val_metrics["f1_score"]:.4f}')
            
            # 简单的早停机制
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                # 可以在这里保存最佳模型
                torch.save(self.model.state_dict(), 'best_model.pth')
        
        print("训练完成!")
        return self.train_losses, self.val_losses
    
    def _calculate_metrics(self, preds, labels, threshold=0.3):
        """计算评估指标"""
        # 将概率转换为二进制预测
        binary_preds = (np.array(preds) > threshold).astype(int)
        binary_labels = np.array(labels)
        
        # 计算Hamming Loss
        h_loss = hamming_loss(binary_labels, binary_preds)
        
        # 计算F1-score (micro average)
        f1 = f1_score(binary_labels, binary_preds, average='micro', zero_division=0)
        
        return {
            'hamming_loss': h_loss,
            'f1_score': f1
        }
    
    def evaluate(self, df, batch_size=16, threshold=0.3):
        """在完整数据集上评估模型"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        descriptions = df['description'].tolist()
        
        skills_list = df['required_skillsets'].apply(
            lambda x: [skill.strip() for skill in str(x).split(',') if skill.strip()]
        ).tolist()
        
        # 转换标签
        y_true = self.mlb.transform(skills_list)
        
        # 创建数据集和数据加载器
        dataset = ProjectDataset(descriptions, y_true, self.tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        
        self.model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.sigmoid(outputs).cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        # 计算评估指标
        metrics = self._calculate_metrics(all_preds, all_labels, threshold)
        
        print("=== 模型评估结果 ===")
        print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")
        print(f"F1-Score: {metrics['f1_score']:.4f}")
        
        return metrics
    
    def predict_skills(self, description, threshold=0.3):
        """预测技能"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        self.model.eval()
        
        encoding = self.tokenizer(
            description,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        
        with torch.no_grad():
            input_ids = encoding['input_ids'].to(self.device)
            attention_mask = encoding['attention_mask'].to(self.device)
            
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            probabilities = torch.sigmoid(outputs).cpu().numpy()[0]
        
        # 获取预测的技能
        predicted_skills = []
        for i, prob in enumerate(probabilities):
            if prob > threshold:
                skill = self.mlb.classes_[i]
                predicted_skills.append((skill, prob))
        
        predicted_skills.sort(key=lambda x: x[1], reverse=True)
        
        return [skill for skill, prob in predicted_skills]
    
    def save_model(self, filepath):
        """保存模型"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'mlb': self.mlb,
            'tokenizer': self.tokenizer,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses
        }, filepath)
        print(f"模型已保存到 {filepath}")
    
    def load_model(self, filepath):
        """加载模型"""
        checkpoint = torch.load(filepath, map_location=self.device, weights_only=False)
        self.mlb = checkpoint['mlb']
        self.tokenizer = checkpoint['tokenizer']
        self.train_losses = checkpoint.get('train_losses', [])
        self.val_losses = checkpoint.get('val_losses', [])
        
        self.model = TransformerSkillPredictor(len(self.mlb.classes_))
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        
        print(f"模型已从 {filepath} 加载")
def get_skills_from_description(descriptions):
    df = pd.read_csv('projects_with_skillsets.csv')
    model_path = 'deep_skill_model.pth'
    if os.path.exists(model_path):
        deep_predictor = DeepSkillPredictor(model_name='distilbert-base-uncased')
        deep_predictor.load_model(model_path)
    skills = []
    for each in descriptions:
        skill = deep_predictor.predict_skills(each, threshold=0.2)
        skills.append(skill)
    return skills

if __name__ == "__main__":
    des = [
        "Develop a cloud-based machine learning platform for financial data analysis",
        "Build a responsive web application with React and JavaScript",
        "Create a data analytics dashboard with real-time visualization"
    ]
    skills = get_skills_from_description(des)
    print(skills)

/userhome/cs/u3658036/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


模型已从 deep_skill_model.pth 加载
[['Python', 'Data Analysis', 'JavaScript', 'Machine Learning', 'AWS', 'React'], ['React', 'JavaScript'], ['Data Analysis', 'Python', 'React', 'JavaScript']]
